# T2 — K-Best Monotonic Temporal Hypotheses

Compare the exact `ORDER_ONLY_MONOTONIC_DP` K=1 baseline with deterministic bounded alternatives at K=3 and K=5. This experiment ends at BTC technical keyframes and reports coarse-window recall, not exact-frame localization.

## INPUT CẦN GẮN TRÊN KAGGLE

1. Stage 1 index bundle: `/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle`
2. Stage 1B encoder verification: `/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports`
3. Stage 1E frozen language path: `/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze`
4. Offline OpenAI CLIP: `/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32`
5. Offline OPUS vi-en translator: `/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en`
6. Frozen RT2 benchmark containing `rt2_ai_benchmark.jsonl`: `/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle`

Nested roots are resolved by marker. Environment overrides are shown in the configuration cell.

## INPUT KHÔNG CẦN

Raw AIC videos/dataset, Stage 0, RT2 evaluation output, M1, MB1/MB1-E1, OCR, ASR, Objects, VLM, Event Graph, and Agent assets. No raw video is opened.

## OUTPUT ZIP

`/kaggle/working/triage_eg_t2_kbest_bundle.zip`


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path("/kaggle/working/AIC2026_TeamPTK_SGU")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True
    )
sys.path.insert(0, str(REPO_DIR / "src"))
COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("resolved commit:", COMMIT)

In [ ]:
STAGE1_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle"
    )
)
STAGE1B_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    )
)
STAGE1E_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1E_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze",
    )
)
CLIP_INPUT = Path(
    os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32")
)
OPUS_INPUT = Path(
    os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en")
)
BENCHMARK_INPUT = Path(
    os.environ.get(
        "AIC_RT2_BENCHMARK_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle",
    )
)
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_t2_kbest")
ZIP_PATH = Path("/kaggle/working/triage_eg_t2_kbest_bundle.zip")
print(
    {
        "stage1": str(STAGE1_INPUT),
        "stage1b": str(STAGE1B_INPUT),
        "stage1e": str(STAGE1E_INPUT),
        "clip": str(CLIP_INPUT),
        "opus": str(OPUS_INPUT),
        "benchmark": str(BENCHMARK_INPUT),
        "output": str(OUTPUT_ROOT),
        "zip": str(ZIP_PATH),
    }
)

In [ ]:
SEARCH_ROOT = Path("/kaggle/input")


def find_marker_roots(root: Path, marker: str, max_depth: int = 6):
    root = Path(root)
    matches, frontier = [], [(root, 0)]
    while frontier:
        current, depth = frontier.pop(0)
        if (current / marker).is_file():
            matches.append(current.resolve())
            continue
        if depth < max_depth and current.is_dir():
            frontier.extend(
                (child, depth + 1) for child in sorted(current.iterdir()) if child.is_dir()
            )
    return sorted(set(matches))


def resolve_root(requested: Path, marker: str) -> Path:
    matches = find_marker_roots(requested, marker)
    if not matches:
        matches = find_marker_roots(SEARCH_ROOT, marker)
    if len(matches) != 1:
        raise RuntimeError(f"Expected one root containing {marker}; found {matches}")
    return matches[0]


def resolve_input_file(requested: Path, filename: str) -> Path:
    if requested.is_file() and requested.name == filename:
        return requested.resolve()
    roots = find_marker_roots(requested, filename)
    if not roots:
        roots = find_marker_roots(SEARCH_ROOT, filename)
    paths = sorted({(root / filename).resolve() for root in roots})
    if len(paths) != 1:
        raise RuntimeError(f"Expected one {filename}; found {paths}")
    return paths[0]


STAGE1_ROOT = resolve_root(STAGE1_INPUT, "stage1_summary.json")
STAGE1B_ROOT = resolve_root(STAGE1B_INPUT, "stage1b_summary.json")
STAGE1E_ROOT = resolve_root(STAGE1E_INPUT, "language_path_contract.json")
CLIP_ROOT = resolve_root(CLIP_INPUT, "checkpoint/ViT-B-32.pt")
OPUS_ROOT = resolve_root(OPUS_INPUT, "model/config.json")
BENCHMARK_PATH = resolve_input_file(BENCHMARK_INPUT, "rt2_ai_benchmark.jsonl")
print(
    json.dumps(
        {
            "stage1_root": str(STAGE1_ROOT),
            "stage1b_root": str(STAGE1B_ROOT),
            "stage1e_root": str(STAGE1E_ROOT),
            "clip_root": str(CLIP_ROOT),
            "opus_root": str(OPUS_ROOT),
            "benchmark_path": str(BENCHMARK_PATH),
        },
        indent=2,
    )
)

In [ ]:
from triage_eg.experiments.temporal_t2 import T2RunnerConfig, T2Settings, preflight_t2
from triage_eg.retrieval.stage2 import config_from_yaml

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
STAGE2 = config_from_yaml(
    REPO_DIR / "configs/retrieval/stage2_operational_runtime.yaml",
    stage1_root=STAGE1_ROOT,
    stage1b_root=STAGE1B_ROOT,
    stage1e_root=STAGE1E_ROOT,
    clip_asset_root=CLIP_ROOT,
    translator_asset_root=OPUS_ROOT,
    output_root=OUTPUT_ROOT / "_stage2_control",
    stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
    build_git_commit=COMMIT,
)
CONFIG = T2RunnerConfig(
    stage2=STAGE2, benchmark_path=BENCHMARK_PATH, output_root=OUTPUT_ROOT, settings=T2Settings()
)
PREFLIGHT = preflight_t2(CONFIG)
print(json.dumps(PREFLIGHT, indent=2))

In [ ]:
from triage_eg.experiments.reference_rt2 import load_rt2_benchmark
from triage_eg.experiments.temporal_t2 import run_t2

QUERIES = load_rt2_benchmark(BENCHMARK_PATH)
RESULT = run_t2(CONFIG, QUERIES)
print(json.dumps(RESULT["summary"], indent=2))

In [ ]:
primary = RESULT["metrics"]["OVERALL"]["PRIMARY_6_SECONDS"]
print("PRIMARY ±6 SECOND COARSE-WINDOW METRICS")
print(json.dumps(primary, indent=2))
print("T2_QUALITY_DECISION = NOT_EVALUATED")

In [ ]:
from triage_eg.experiments.temporal_t2 import create_t2_bundle

bundle = create_t2_bundle(OUTPUT_ROOT, ZIP_PATH)
print("T2_IMPLEMENTATION_STATUS = COMPLETE")
print("T2_REAL_STATUS = COMPLETE")
print("T2_QUALITY_DECISION = NOT_EVALUATED")
print("DOWNLOAD ZIP:", bundle, "size_bytes=", bundle.stat().st_size)